# anekamacam datagen on Colab

Generate a Texel-tuning dataset by parallel self-play, then optionally tune.

**Use a CPU runtime (High-RAM), NOT a GPU/TPU runtime.** The engine is pure
handcrafted eval — search, datagen, and the tuner are all CPU-bound. A GPU
accelerates nothing here and burns compute units fast. Runtime → Change
runtime type → CPU. On Colab Pro, enable background execution for long runs.

Throughput = number of parallel processes (games play sequentially inside a
process). Each shard runs single-thread in its own directory; `merge_shards.py`
reindexes colliding game IDs into one game-disjoint corpus.

In [ ]:
# 1. Install the nightly Rust toolchain (engine uses #![feature(...)]).
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain nightly
import os
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
!rustc --version

In [ ]:
# 2. Clone the repo. For a private repo, use a token or the git URL you have.
REPO_URL = 'https://github.com/aldenluthfi/anekamacam.git'  # <-- edit
!rm -rf anekamacam && git clone --depth 1 $REPO_URL
%cd anekamacam

In [ ]:
# 3. Build the release binary (~1-3 min).
!cargo build --release
!./target/release/anekamacam derive >/dev/null 2>&1 && echo 'build + derive ok'

In [ ]:
# 4. Parameters, then launch one single-thread shard per core.
VARIANT   = 'standard'   # standard | shogi | xiangqi | crazyhouse | ...
GAMES     = 200      # games PER shard
MOVETIME  = 100      # ms per move

import os, subprocess
SHARDS = os.cpu_count()
print(f'{SHARDS} shards x {GAMES} games ({VARIANT}, {MOVETIME}ms/move)')

procs = []
for k in range(SHARDS):
    d = f'shard_{k}'
    os.makedirs(d, exist_ok=True)
    procs.append(subprocess.Popen(
        ['../target/release/anekamacam', 'datagen', VARIANT,
         str(GAMES), str(MOVETIME), '1'],
        cwd=d, stdin=subprocess.DEVNULL,
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL))
for p in procs:
    p.wait()
print('all shards done')

In [ ]:
# 5. Reindex + merge shards into res/data/<variant>/latest.data.
!python3 tools/merge_shards.py $VARIANT
!rm -rf shard_*

In [ ]:
# 6. (Optional) Texel-tune from the merged dataset. Logs land in logs/.
EPOCHS = 200
!./target/release/anekamacam tune $VARIANT $EPOCHS
!grep -iE 'Tune split|Tune results|epoch|Tune complete' $(ls -t logs/latest-*.log | head -1) | tail -15

In [ ]:
# 7. Download the dataset and any tuned params.
import shutil
shutil.make_archive('datagen_output', 'zip', '.', 'res/data')
shutil.make_archive('param_output', 'zip', 'res/param', VARIANT)
from google.colab import files
files.download('datagen_output.zip')
files.download('param_output.zip')